Импорты

In [1]:
import numpy as np
import librosa
import scipy.signal as signal
import matplotlib.pyplot as plt
from IPython.display import Audio

Загрузка аудиофайла

In [10]:
X, original_sr = librosa.load('/content/lana_del_rej_summertime_sadness.mp3', sr=None)
test_audio = X[int(original_sr*5):int(original_sr*7)]

print("ОРИГИНАЛ:")
print(f"Частота: {original_sr} Гц, Длительность: {len(test_audio)/original_sr:.2f} сек")
display(Audio(test_audio, rate=original_sr))

/tmp/ipython-input-2332063003.py:1: UserWarning: PySoundFile failed. Trying audioread instead.
  X, original_sr = librosa.load('/content/lana_del_rej_summertime_sadness.mp3', sr=None)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


ОРИГИНАЛ:
Частота: 44100 Гц, Длительность: 2.00 сек


Создание фильтров

In [ ]:
def make_fir_filter(cutoff=0.4):
    return signal.firwin(101, cutoff, window='hamming')

def make_iir_filter(cutoff=0.4):
    b, a = signal.butter(6, cutoff, btype='low')
    return b, a

*FIR фильтр создает 101 коэффициент для "взвешивания" отсчетов, а IIR фильтр создает 6-го порядка рекурсивную формулу с обратными связями, оба обрезая частоты выше 0.4 от частоты Найквиста.*

Реализация частотного конвертера для случая N1 > N2

In [11]:
def manual_downsample_fixed(audio, sr_orig, sr_target, filter_type='fir'):

    # вычисление точного кол-ва отсчетов для сохранения длительности
    duration = len(audio) / sr_orig
    target_samples = int(duration * sr_target)

    # коэффициент децимации
    ratio = sr_orig / sr_target

    #  фильтр с правильной частотой среза
    cutoff = 0.9 / ratio  # частота среза = 0.9 * новая_частота_Найквиста
    fir_taps = make_fir_filter(cutoff)
    iir_b, iir_a = make_iir_filter(cutoff)

    # антиалиасинговая фильтрация
    if filter_type == 'fir':
        filtered = signal.lfilter(fir_taps, 1.0, audio)
    else:
        filtered = signal.filtfilt(iir_b, iir_a, audio)

    # линейная интерполяция для точного ресемплинга
    x_old = np.linspace(0, 1, len(filtered))
    x_new = np.linspace(0, 1, target_samples)

    result = np.interp(x_new, x_old, filtered)

    return result

Частотный конвертер понижения, который преобразует аудиосигнал из одной частоты дискретизации в другую через три ключевых этапа:

- антиалиасинговую фильтрацию (FIR/IIR фильтр отсекает высокие частоты выше новой частоты Найквиста),

- пересчёт количества отсчетов (точное вычисление target_samples для сохранения длительности)

- линейную интерполяцию (плавное преобразование в новую временную сетку).

Реализация частотного конвертера для случая N1 < N2

In [12]:
def manual_upsample_fixed(audio, sr_orig, sr_target, filter_type='fir'):

    # вычисление точного кол-ва отсчетов для сохранения длительности
    duration = len(audio) / sr_orig
    target_samples = int(duration * sr_target)

    # коэффициент интерполяции
    ratio = sr_target / sr_orig

    # фильтр с правильной частотой среза
    cutoff = 0.9 / ratio  # Частота среза = 0.9 * исходная_частота_Найквиста
    fir_taps = make_fir_filter(cutoff)
    iir_b, iir_a = make_iir_filter(cutoff)

    # интерполяция до нужного кол-ва отсчетов
    x_old = np.linspace(0, 1, len(audio))
    x_new = np.linspace(0, 1, target_samples)
    upsampled = np.interp(x_new, x_old, audio)

    # антиалиасинговая фильтрация
    if filter_type == 'fir':
        result = signal.lfilter(fir_taps, 1.0, upsampled)
    else:
        result = signal.filtfilt(iir_b, iir_a, upsampled)

    return result

*Повышает частоту дискретизации аудио путём линейной интерполяции до target_samples отсчётов для сохранения длительности и последующей FIR/IIR фильтрации для удаления спектральных копий.*

Сигналы нужных частот с правильной длительностью

In [13]:
duration = len(test_audio) / original_sr

audio_22050 = signal.resample(test_audio, int(duration * 22050))
audio_8000 = signal.resample(test_audio, int(duration * 8000))

print(f"audio_22050: {len(audio_22050)} отсчетов, {len(audio_22050)/22050:.3f} сек")
print(f"audio_8000: {len(audio_8000)} отсчетов, {len(audio_8000)/8000:.3f} сек")

audio_22050: 44100 отсчетов, 2.000 сек
audio_8000: 16000 отсчетов, 2.000 сек


ПОНИЖЕНИЕ 22050 → 16000

FIR фильтр

In [16]:
my_down_fir = manual_downsample_fixed(audio_22050, 22050, 16000, 'fir')
display(Audio(my_down_fir, rate=16000))

IIR фильтр

In [17]:
my_down_iir = manual_downsample_fixed(audio_22050, 22050, 16000, 'iir')
display(Audio(my_down_iir, rate=16000))

Анализ:


*По звучанию фильтры между собой звучат идентично. Приглушенный звук, мелодия уходит на задний план. Явных дефектов нет.*

ПОВЫШЕНИЕ 8000 → 22050

FIR фильтр

In [18]:
my_up_fir = manual_upsample_fixed(audio_8000, 8000, 22050, 'fir')
display(Audio(my_up_fir, rate=22050))

IIR фильтр

In [19]:
my_up_iir = manual_upsample_fixed(audio_8000, 8000, 22050, 'iir')
display(Audio(my_up_iir, rate=22050))

Анализ:

*Фильтры тоже идентичны по звучанию между собой. В сравнении с понижением, на задний план уходит и мелодия и голос. Звук приглушенный. Явных дефектов не слышно.*

СРАВНЕНИЕ С LIBROSA

Librosa понижение 22050 → 16000

In [20]:
librosa_down = librosa.resample(audio_22050, orig_sr=22050, target_sr=16000)
display(Audio(librosa_down, rate=16000))

Librosa повышение 8000 → 22050

In [21]:
librosa_up = librosa.resample(audio_8000, orig_sr=8000, target_sr=22050)
display(Audio(librosa_up, rate=22050))

Анализ:

*Либроса звучит точно так же как и реализованные выше конверторы.*